[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/074.%20The%20VRP%20with%20Backhauls%20%28VRPB%29/P74-Tier-7_executed.ipynb)

---


# 74. The Vehicle Routing Problem with Backhauls: Human–AI Collaboration Simulation

## Purpose and scope

A vehicle routing problem with backhauls (VRPB) combines delivery customers, where goods are unloaded, with pickup customers, where goods are collected. A VRPB routing model normally needs customer locations, demands, vehicle capacities, travel costs, and rules governing the order in which deliveries and pickups may be visited.

This notebook focuses on a different, narrower question: how can a human routing expert and an AI recommendation system interact, record decisions, and update a simple trust score? The code simulates that collaboration process. It does **not** solve a VRPB: it has no customer or road-network data, no vehicle-capacity calculation, and no route optimization. The displayed route sequence is illustrative, not a verified feasible or optimal VRPB route. Synthetic costs, satisfaction scores, emissions, and interaction outcomes are used to demonstrate the collaboration workflow rather than to make claims about real operations.

The notebook follows this sequence: define expert and recommendation records, define trust updates and a collaboration session, assemble the partnership system, simulate sessions, summarize the recorded outcomes, and visualize the results. The simulation uses Python's `random` module without setting a seed, so the values and decisions can differ between runs.

## Tier 7: Human–AI symbiotic partnership

This tier emphasizes explainable recommendations, human oversight, trust, and feedback. The simulation represents these ideas with explicit reasoning text, expert-specific decision rules, recorded acceptance or modification decisions, and trust metrics. These are illustrative mechanisms; they are not a validated psychological model or a learned AI system.

The four expert profiles represent different priorities: cost, driver satisfaction, environmental impact, and customer service. Their decision styles and experience are recorded as profile information. In this implementation, decision rules use expertise area and, in selected cases, decision style; experience and job satisfaction do not affect the simulated decisions or trust updates.

## Model assumptions and mathematical rules

Each collaboration session has a synthetic AI recommendation, a human acceptance or modification decision, and an outcome. The AI's cost, driver-satisfaction score, and emissions are sampled independently from uniform ranges. For example, `random.uniform(700, 950)` samples a cost between 700 and 950, including possible floating-point values between the endpoints. The route sequence is fixed for demonstration and is not derived from those sampled metrics.

Trust has four components: competence, integrity, benevolence, and predictability. Each component is bounded between 0 and 1. For a component \(T\), the update rule is

$$
T' = \begin{cases}
\min(1, T + \delta), & \text{if the corresponding positive interaction condition holds},\\
\max(0, T - \delta), & \text{if the competence outcome is unsuccessful},\\
T, & \text{if no update is specified for that component}.
\end{cases}
$$

The increments δ are 0.05 for competence, 0.04 for integrity, 0.06 for benevolence, and 0.03 for predictability. Competence decreases by 0.03 when an outcome is unsuccessful. The overall trust score is the arithmetic mean of the four components:

$$
\tau = \frac{T_{\mathrm{competence}} + T_{\mathrm{integrity}} + T_{\mathrm{benevolence}} + T_{\mathrm{predictability}}}{4}.
$$

The code implements these rules in `TrustMetrics.update_trust`. A session is marked successful when its final cost does not exceed its budget limit. The simulation's acceptance rate, modification rate, and success rate are descriptive proportions over the simulated sessions; they are not measures of VRPB solution quality.

The recommendation confidence is a demonstration score, not a statistically calibrated probability. In code it is calculated as

$$
c = \max\left(0.6,\;\min\left(0.95,\;0.9 - 0.05n\right)\right),
$$

where \(n\) is the number of entries in `context['constraints']`, or zero if that key is absent. The simulated session contexts do not provide that key, so the formula uses \(n=0\). The stored score is a fraction on the 0-to-1 scale; its explanation displays it as a percentage.

## Interpreting the results

Use the printed summary and plots to inspect what happened in this particular stochastic run: how the modeled trust scores changed, how often recommendations were accepted, and what costs and satisfaction scores the simulation produced. A higher trust score means only that the specified update rules increased the score; it does not establish that a recommendation was correct, that a human trusted the system in a measured real-world sense, or that the route is feasible. Repeated runs can differ because no random seed is set.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
import random
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Consistent styling makes comparisons across the four result panels easier to read.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for the human-AI collaboration simulation.")

In [ ]:
@dataclass
class HumanExpert:
    """Store an expert profile and apply the simulation's decision rules."""

    name: str
    expertise_area: str
    experience_years: int
    decision_style: str
    trust_score: float = 0.5
    satisfaction_score: float = 0.7
    collaboration_count: int = 0

    def evaluate_ai_recommendation(
        self, ai_route: Dict[str, Any], context: Dict[str, Any]
    ) -> Tuple[bool, str]:
        """Apply profile-specific rules and return an acceptance decision and reason."""
        self.collaboration_count += 1

        if self.expertise_area == "driver_satisfaction":
            if (
                ai_route["driver_satisfaction"] < 80
                and self.decision_style == "conservative"
            ):
                return (
                    False,
                    f"Driver satisfaction is below the threshold "
                    f"({ai_route['driver_satisfaction']:.1f}%).",
                )
        elif self.expertise_area == "cost":
            if (
                ai_route["cost"] > context["budget_limit"]
                and self.decision_style == "aggressive"
            ):
                return (
                    False,
                    f"Cost exceeds the budget "
                    f"(${ai_route['cost']:.2f} > ${context['budget_limit']:.2f}).",
                )

        # The low-trust rule introduces a stochastic decision when trust is below 0.3.
        if self.trust_score < 0.3 and random.random() < 0.7:
            return False, "The expert does not yet trust the AI recommendation."

        return True, "The AI recommendation is accepted."

    def modify_route(
        self, ai_route: Dict[str, Any], context: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Apply the illustrative metric changes associated with an expert's specialty."""
        modified_route = ai_route.copy()

        if self.expertise_area == "driver_satisfaction":
            modified_route["driver_satisfaction"] = min(
                95,
                ai_route["driver_satisfaction"] + random.randint(10, 25),
            )
            modified_route["cost"] *= 1 + random.uniform(0.05, 0.15)
        elif self.expertise_area == "cost":
            modified_route["cost"] *= 1 - random.uniform(0.03, 0.12)
            modified_route["driver_satisfaction"] *= 1 - random.uniform(0.02, 0.08)
        elif self.expertise_area == "environmental":
            modified_route["emissions"] *= 1 - random.uniform(0.10, 0.25)
            modified_route["cost"] *= 1 + random.uniform(0.02, 0.08)

        modified_route["modified_by"] = self.name
        modified_route["modification_reason"] = (
            f"{self.expertise_area} optimization"
        )
        return modified_route


@dataclass
class AIRecommendation:
    """Store an illustrative recommendation and its human-readable explanation."""

    route_sequence: List[int]
    cost: float
    driver_satisfaction: float
    emissions: float
    confidence: float
    reasoning: List[str]
    alternatives: List[Dict[str, Any]] = field(default_factory=list)

    def explain_recommendation(self) -> str:
        """Format the recommendation details for display in a session."""
        explanation = (
            f"Illustrative route sequence: "
            f"{' -> '.join(map(str, self.route_sequence))}\n"
        )
        explanation += (
            f"Synthetic cost: ${self.cost:.2f}; "
            f"driver-satisfaction score: {self.driver_satisfaction:.1f}%; "
            f"emissions: {self.emissions:.1f} kg CO2\n"
        )
        explanation += f"Demonstration confidence score: {self.confidence * 100:.1f}%\n"
        explanation += "Simulation notes:\n" + "\n".join(
            f"- {reason}" for reason in self.reasoning
        )
        return explanation


print("Expert and recommendation classes are defined.")

In [ ]:
@dataclass
class TrustMetrics:
    """Track four bounded trust components and their arithmetic mean."""

    competence_trust: float = 0.5
    integrity_trust: float = 0.5
    benevolence_trust: float = 0.5
    predictability_trust: float = 0.5
    overall_trust: float = 0.5

    def update_trust(
        self, interaction_outcome: Dict[str, Any], human_expert: HumanExpert
    ) -> None:
        """Update trust components from the recorded outcome and sync the expert score."""
        if interaction_outcome["success"]:
            self.competence_trust = min(1.0, self.competence_trust + 0.05)
        else:
            self.competence_trust = max(0.0, self.competence_trust - 0.03)

        if interaction_outcome.get("explanation_clear", False):
            self.integrity_trust = min(1.0, self.integrity_trust + 0.04)

        if interaction_outcome.get("human_beneficial", False):
            self.benevolence_trust = min(1.0, self.benevolence_trust + 0.06)

        if interaction_outcome.get("consistent", False):
            self.predictability_trust = min(1.0, self.predictability_trust + 0.03)

        self.overall_trust = (
            self.competence_trust
            + self.integrity_trust
            + self.benevolence_trust
            + self.predictability_trust
        ) / 4
        human_expert.trust_score = self.overall_trust


@dataclass
class CollaborationSession:
    """Represent one recommendation, human decision, and recorded outcome."""

    session_id: int
    human_expert: HumanExpert
    ai_recommendation: AIRecommendation
    context: Dict[str, Any]
    timestamp: datetime = field(default_factory=datetime.now)
    outcome: Optional[Dict[str, Any]] = None

    def execute_collaboration(self) -> Dict[str, Any]:
        """Run the decision rules and return the session outcome."""
        print(f"\n[SESSION {self.session_id}] Expert: {self.human_expert.name}")
        print(self.ai_recommendation.explain_recommendation())

        ai_route = {
            "cost": self.ai_recommendation.cost,
            "driver_satisfaction": self.ai_recommendation.driver_satisfaction,
            "emissions": self.ai_recommendation.emissions,
        }
        accept, reason = self.human_expert.evaluate_ai_recommendation(
            ai_route, self.context
        )
        decision = "ACCEPT" if accept else "MODIFY"
        print(f"Human decision: {decision} — {reason}")

        if accept:
            final_route = {
                "route_sequence": self.ai_recommendation.route_sequence,
                **ai_route,
                "accepted": True,
            }
        else:
            final_route = self.human_expert.modify_route(
                {
                    "route_sequence": self.ai_recommendation.route_sequence,
                    **ai_route,
                },
                self.context,
            )
            final_route["accepted"] = False
            print(
                f"Modified synthetic metrics — cost: ${final_route['cost']:.2f}; "
                f"driver-satisfaction score: "
                f"{final_route['driver_satisfaction']:.1f}%"
            )

        outcome = {
            "final_route": final_route,
            "success": final_route["cost"]
            <= self.context.get("budget_limit", 1000),
            "explanation_clear": bool(self.ai_recommendation.reasoning),
            "human_beneficial": final_route["driver_satisfaction"] > 75,
            "consistent": random.random() > 0.2,
            "ai_accepted": accept,
            "human_expert": self.human_expert.name,
        }
        self.outcome = outcome
        return outcome


print("Trust metrics and collaboration session classes are defined.")

In [ ]:
class HumanAISymbioticPartnership:
    """Manage expert profiles, simulated sessions, and per-expert trust records."""

    def __init__(self) -> None:
        self.human_experts: List[HumanExpert] = []
        self.trust_metrics: Dict[str, TrustMetrics] = {}
        self.collaboration_history: List[CollaborationSession] = []
        self.session_counter = 0

    def add_human_expert(self, expert: HumanExpert) -> None:
        """Register an expert and initialize that expert's trust components."""
        self.human_experts.append(expert)
        self.trust_metrics[expert.name] = TrustMetrics()

    def generate_ai_recommendation(
        self, context: Dict[str, Any]
    ) -> AIRecommendation:
        """Sample synthetic metrics and create an illustrative recommendation record."""
        base_cost = random.uniform(700, 950)
        base_satisfaction = random.uniform(60, 85)
        base_emissions = random.uniform(80, 120)

        # This fixed sequence illustrates a displayed route; no routing model evaluates it.
        route_sequence = [0, 1, 3, 2, 4, 5, 6, 7, 0]
        reasoning = [
            f"Synthetic baseline cost sampled as ${base_cost:.2f}.",
            f"Synthetic driver-satisfaction score sampled as {base_satisfaction:.1f}%.",
            f"Synthetic emissions value sampled as {base_emissions:.1f} kg CO2.",
            "The route sequence is illustrative and has not been checked for feasibility.",
            "No vehicle-capacity or travel-cost calculation is performed.",
        ]

        constraint_count = len(context.get("constraints", []))
        confidence = max(
            0.6,
            min(0.95, 0.9 - constraint_count * 0.05),
        )

        return AIRecommendation(
            route_sequence=route_sequence,
            cost=base_cost,
            driver_satisfaction=base_satisfaction,
            emissions=base_emissions,
            confidence=confidence,
            reasoning=reasoning,
        )

    def conduct_collaboration_session(
        self, expert_name: str, context: Dict[str, Any]
    ) -> CollaborationSession:
        """Create, execute, record, and learn from one collaboration session."""
        expert = next(
            (candidate for candidate in self.human_experts if candidate.name == expert_name),
            None,
        )
        if expert is None:
            raise ValueError(f"Expert {expert_name!r} is not registered.")

        self.session_counter += 1
        ai_recommendation = self.generate_ai_recommendation(context)
        session = CollaborationSession(
            session_id=self.session_counter,
            human_expert=expert,
            ai_recommendation=ai_recommendation,
            context=context,
        )
        outcome = session.execute_collaboration()
        self.trust_metrics[expert.name].update_trust(outcome, expert)
        self.collaboration_history.append(session)
        return session

    def run_partnership_simulation(
        self, num_sessions: int = 20
    ) -> List[CollaborationSession]:
        """Run the requested number of randomly generated collaboration sessions."""
        if not self.human_experts:
            raise ValueError("Add at least one human expert before running sessions.")

        sessions = []
        for _ in range(num_sessions):
            expert = random.choice(self.human_experts)
            context = {
                "budget_limit": random.uniform(800, 1000),
                "urgent_delivery": random.random() > 0.7,
                "weather_conditions": random.choice(["clear", "rain", "snow"]),
                "driver_availability": random.choice(["high", "medium", "low"]),
            }
            sessions.append(
                self.conduct_collaboration_session(expert.name, context)
            )
        return sessions


print("The human-AI partnership simulation system is defined.")

In [ ]:
# The profiles preserve distinct specialties and decision styles for the simulation.
experts = [
    HumanExpert(
        name="Sarah Chen",
        expertise_area="driver_satisfaction",
        experience_years=12,
        decision_style="balanced",
    ),
    HumanExpert(
        name="Marcus Rodriguez",
        expertise_area="cost",
        experience_years=8,
        decision_style="aggressive",
    ),
    HumanExpert(
        name="Dr. Emily Watson",
        expertise_area="environmental",
        experience_years=15,
        decision_style="conservative",
    ),
    HumanExpert(
        name="James Kim",
        expertise_area="customer_service",
        experience_years=6,
        decision_style="balanced",
    ),
]

partnership = HumanAISymbioticPartnership()
for expert in experts:
    partnership.add_human_expert(expert)

# Retain each starting value so the summary reports changes from the actual setup state.
initial_trust = {expert.name: expert.trust_score for expert in experts}

print(
    f"Partnership initialized with {len(experts)} expert profiles."
)
print("\nExpert profiles:")
for expert in experts:
    print(
        f"- {expert.name}: {expert.expertise_area} specialist, "
        f"{expert.experience_years} years of experience"
    )
    print(
        f"  Decision style: {expert.decision_style}; "
        f"initial trust score: {expert.trust_score:.2f}"
    )

In [ ]:
print("\n" + "=" * 60)
print("Starting the human-AI collaboration simulation")
print("=" * 60)

# Twenty sessions retain the example's specified simulation length.
sessions = partnership.run_partnership_simulation(num_sessions=20)
print(f"\nSimulation completed: {len(sessions)} sessions recorded.")

print("\n" + "=" * 60)
print("Simulation summary")
print("=" * 60)

print("\nTrust scores by expert:")
for expert_name, trust_metrics in partnership.trust_metrics.items():
    expert = next(item for item in experts if item.name == expert_name)
    start = initial_trust[expert_name]
    print(f"\n{expert_name}:")
    print(f"  Initial trust score: {start:.2f}")
    print(f"  Final trust score: {trust_metrics.overall_trust:.3f}")
    print(f"  Change in trust score: {trust_metrics.overall_trust - start:+.3f}")
    print(f"  Recorded collaborations: {expert.collaboration_count}")
    print(
        f"  Stored job-satisfaction score (not updated by this model): "
        f"{expert.satisfaction_score:.2f}"
    )

ai_accepted = sum(
    session.outcome["ai_accepted"] for session in sessions
)
human_modified = len(sessions) - ai_accepted
successful_outcomes = sum(
    session.outcome["success"] for session in sessions
)

print("\nDecision and budget-threshold summaries:")
print(
    f"  Recommendations accepted: {ai_accepted}/{len(sessions)} "
    f"({ai_accepted / len(sessions) * 100:.1f}%)"
)
print(
    f"  Recommendations modified: {human_modified}/{len(sessions)} "
    f"({human_modified / len(sessions) * 100:.1f}%)"
)
print(
    f"  Final costs at or below their session budget: "
    f"{successful_outcomes}/{len(sessions)} "
    f"({successful_outcomes / len(sessions) * 100:.1f}%)"
)

average_cost = np.mean(
    [session.outcome["final_route"]["cost"] for session in sessions]
)
average_satisfaction = np.mean(
    [
        session.outcome["final_route"]["driver_satisfaction"]
        for session in sessions
    ]
)
print("\nSynthetic final-metric averages:")
print(f"  Average cost: ${average_cost:.2f}")
print(f"  Average driver-satisfaction score: {average_satisfaction:.1f}%")
print("  These values describe this simulation run, not optimized routing performance.")

In [ ]:
def visualize_partnership_results(
    sessions: List[CollaborationSession],
    partnership: HumanAISymbioticPartnership,
) -> None:
    """Plot trust history, acceptance, outcome metrics, and final trust components."""
    if not sessions:
        raise ValueError("At least one session is needed to create the summary plots.")

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(16, 12)
    )
    fig.suptitle(
        "Human-AI Collaboration Simulation", fontsize=16, fontweight="bold"
    )

    # Reconstruct each expert's trust history locally; this leaves recorded final state unchanged.
    ax1.set_title("Modeled Trust Score by Collaboration")
    for expert in partnership.human_experts:
        expert_sessions = [
            session
            for session in sessions
            if session.outcome["human_expert"] == expert.name
        ]
        if not expert_sessions:
            continue

        temporary_trust = TrustMetrics()
        temporary_expert = HumanExpert(
            name=expert.name,
            expertise_area=expert.expertise_area,
            experience_years=expert.experience_years,
            decision_style=expert.decision_style,
        )
        trust_values = [temporary_trust.overall_trust]
        for session in expert_sessions:
            temporary_trust.update_trust(session.outcome, temporary_expert)
            trust_values.append(temporary_trust.overall_trust)

        ax1.plot(
            range(len(trust_values)),
            trust_values,
            marker="o",
            label=expert.name,
            linewidth=2,
            markersize=5,
        )
    ax1.set_xlabel("Expert-specific collaboration number")
    ax1.set_ylabel("Trust score (0 to 1)")
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)
    ax1.legend(title="Expert")

    acceptance_data = {}
    for expert in partnership.human_experts:
        expert_sessions = [
            session
            for session in sessions
            if session.outcome["human_expert"] == expert.name
        ]
        if expert_sessions:
            accepted = sum(
                session.outcome["ai_accepted"] for session in expert_sessions
            )
            acceptance_data[expert.name] = accepted / len(expert_sessions)

    names = list(acceptance_data)
    values = list(acceptance_data.values())
    ax2.bar(names, values, color="steelblue")
    ax2.set_title("AI Recommendation Acceptance Rate")
    ax2.set_ylabel("Acceptance rate (proportion)")
    ax2.set_ylim(0, 1.12)
    ax2.tick_params(axis="x", rotation=25)
    ax2.grid(True, axis="y", alpha=0.3)
    for index, value in enumerate(values):
        ax2.text(
            index,
            value + 0.02,
            f"{value * 100:.1f}%",
            ha="center",
            va="bottom",
        )

    # These measures use different units; their bar heights should not be compared as a common scale.
    metric_names = []
    cost_efficiency = []
    satisfaction_scores = []
    success_rates = []
    for expert in partnership.human_experts:
        expert_sessions = [
            session
            for session in sessions
            if session.outcome["human_expert"] == expert.name
        ]
        if not expert_sessions:
            continue

        mean_cost = np.mean(
            [session.outcome["final_route"]["cost"] for session in expert_sessions]
        )
        mean_satisfaction = np.mean(
            [
                session.outcome["final_route"]["driver_satisfaction"]
                for session in expert_sessions
            ]
        )
        success_rate = np.mean(
            [session.outcome["success"] for session in expert_sessions]
        )
        metric_names.append(expert.name.split()[0])
        cost_efficiency.append(1000 - mean_cost)
        satisfaction_scores.append(mean_satisfaction)
        success_rates.append(success_rate * 100)

    x = np.arange(len(metric_names))
    width = 0.25
    ax3.bar(
        x - width,
        cost_efficiency,
        width,
        label="Cost index: $1,000 minus mean cost",
    )
    ax3.bar(
        x,
        satisfaction_scores,
        width,
        label="Mean satisfaction score (%)",
    )
    ax3.bar(
        x + width,
        success_rates,
        width,
        label="Budget-threshold rate (%)",
    )
    ax3.set_title("Synthetic Session Metrics by Expert")
    ax3.set_ylabel("Metric value; units differ by measure")
    ax3.set_xticks(x)
    ax3.set_xticklabels(metric_names)
    ax3.legend(fontsize=8)
    ax3.grid(True, axis="y", alpha=0.3)

    trust_components = np.array(
        [
            [
                metrics.competence_trust,
                metrics.integrity_trust,
                metrics.benevolence_trust,
                metrics.predictability_trust,
            ]
            for metrics in partnership.trust_metrics.values()
        ]
    ).T
    expert_labels = [
        expert.name.split()[0] for expert in partnership.human_experts
    ]
    image = ax4.imshow(
        trust_components, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1
    )
    ax4.set_title("Final Modeled Trust Components")
    ax4.set_xlabel("Expert")
    ax4.set_xticks(range(len(expert_labels)))
    ax4.set_xticklabels(expert_labels)
    ax4.set_yticks(range(4))
    ax4.set_yticklabels(
        ["Competence", "Integrity", "Benevolence", "Predictability"]
    )
    for row in range(4):
        for column in range(len(expert_labels)):
            ax4.text(
                column,
                row,
                f"{trust_components[row, column]:.2f}",
                ha="center",
                va="center",
                color="black",
            )
    fig.colorbar(image, ax=ax4, label="Trust score (0 to 1)")

    fig.tight_layout()
    plt.show()


visualize_partnership_results(sessions, partnership)

## Summary and conclusions

The notebook defines and runs a stochastic human–AI collaboration simulation. Its visible steps are expert selection, synthetic recommendation generation, rule-based acceptance or modification, outcome recording, trust-score updates, and summary visualization. The trust score is an arithmetic mean of four components whose changes follow the stated update rules.

Interpret the run-specific values as consequences of those rules and random samples. Acceptance is not evidence that a route is feasible or optimal; the budget-threshold success flag checks only whether the final synthetic cost is within that session's budget. The plotted cost index, satisfaction score, and success rate also have different units and should not be compared as if they shared a common scale.

The simulation is useful for tracing how interaction rules and feedback affect recorded decisions and modeled trust. It does not establish that human–AI collaboration improves routing outcomes, that trust develops at a particular rate, or that the displayed sequence solves a VRPB. A VRPB application would require an explicit routing model and operational data; those elements are outside this notebook's scope.